<a href="https://colab.research.google.com/github/MonKhach/Data_Visualization_project/blob/main/Placement_Data_Full_Class_part_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##In this section, I aim to answer two main questions:

1. Which features are most strongly associated with placement?
2. Among placed students, which factors are related to salary?

The story of this dataset is that placement seems to depend more on academic consistency, work experience, and specialisation than on MBA score alone. Then, among students who are placed, salary differences can be explored through work experience, degree type, specialisation, and employability test performance.

In [34]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [35]:
px.defaults.template = "plotly_dark"

In [36]:
df = pd.read_csv('Placement_Data_Full_Class.csv')
df.head(10)

,sl_no,gender,ssc_p,ssc_b,hsc_p,hsc_b,hsc_s,degree_p,degree_t,workex,etest_p,specialisation,mba_p,status,salary
0,1,M,67.00,Others,91.00,Others,Commerce,58.00,Sci&Tech,No,55.00,Mkt&HR,58.80,Placed,270000.0
1,2,M,79.33,Central,78.33,Others,Science,77.48,Sci&Tech,Yes,86.50,Mkt&Fin,66.28,Placed,200000.0
2,3,M,65.00,Central,68.00,Central,Arts,64.00,Comm&Mgmt,No,75.00,Mkt&Fin,57.80,Placed,250000.0
3,4,M,56.00,Central,52.00,Central,Science,52.00,Sci&Tech,No,66.00,Mkt&HR,59.43,Not Placed,NaN
4,5,M,85.80,Central,73.60,Central,Commerce,73.30,Comm&Mgmt,No,96.80,Mkt&Fin,55.50,Placed,425000.0
5,6,M,55.00,Others,49.80,Others,Science,67.25,Sci&Tech,Yes,55.00,Mkt&Fin,51.58,Not Placed,NaN
6,7,F,46.00,Others,49.20,Others,Commerce,79.00,Comm&Mgmt,No,74.28,Mkt&Fin,53.29,Not Placed,NaN
7,8,M,82.00,Central,64.00,Central,Science,66.00,Sci&Tech,Yes,67.00,Mkt&Fin,62.14,Placed,252000.0
8,9,M,73.00,Central,79.00,Central,Commerce,72.00,Comm&Mgmt,No,91.34,Mkt&Fin,61.29,Placed,231000.0
9,10,M,58.00,Central,70.00,Central,Commerce,61.00,Comm&Mgmt,No,54.00,Mkt&Fin,52.21,Not Placed,NaN


In [37]:
df = df.drop(columns=["sl_no"]).copy()
df["placed"] = df["status"].map({"Placed": 1, "Not Placed": 0})

placed_df = df[df["status"] == "Placed"].copy()

labels = {
    "ssc_p": "Secondary school %",
    "hsc_p": "Higher secondary %",
    "degree_p": "Undergraduate degree %",
    "etest_p": "Employability test %",
    "mba_p": "MBA %",
    "workex": "Work experience",
    "specialisation": "Specialisation",
    "degree_t": "Degree type",
    "hsc_s": "HSC stream"
}

score_cols = ["ssc_p", "hsc_p", "degree_p", "etest_p", "mba_p"]

In [38]:
placement_summary = df.groupby("status")[["ssc_p", "hsc_p", "degree_p", "etest_p", "mba_p"]].mean().round(2)
placement_summary

,ssc_p,hsc_p,degree_p,etest_p,mba_p
status,,,,,
Not Placed,57.54,58.40,61.13,69.59,61.61
Placed,71.72,69.93,68.74,73.24,62.58


In [39]:
df.groupby("workex")["placed"].agg(["mean", "count"]).assign(mean=lambda x: (x["mean"]*100).round(2))

,mean,count
workex,,
No,59.57,141
Yes,86.49,74


In [40]:
overall = df["status"].value_counts().reset_index()
overall.columns = ["Status", "Count"]
overall["Percent"] = (overall["Count"] / overall["Count"].sum() * 100).round(1)

fig = px.bar(
    overall,
    x="Status",
    y="Count",
    text="Percent",
    title="Overall placement outcome"
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.show()

The dataset is imbalanced toward placed students, so most students were successfully placed. This gives us a good basis to study what differentiates placed and not placed students.

In [41]:
band_labels = ["<50", "50-60", "60-70", "70-80", "80+"]

for col in score_cols:
    df[col + "_band"] = pd.cut(
        df[col],
        bins=[0, 50, 60, 70, 80, 100],
        labels=band_labels,
        include_lowest=True,
        right=False
    )

rate_frames = []

for col in score_cols:
    temp = (
        df.groupby(col + "_band", observed=False)["placed"]
        .agg(["mean", "count"])
        .reset_index()
        .rename(columns={
            col + "_band": "Score band",
            "mean": "Placement rate",
            "count": "Students"
        })
    )
    temp["Feature"] = labels[col]
    temp["Placement rate"] = temp["Placement rate"] * 100
    rate_frames.append(temp)

rate_df = pd.concat(rate_frames, ignore_index=True)

fig = px.bar(
    rate_df,
    x="Score band",
    y="Placement rate",
    facet_col="Feature",
    facet_col_wrap=3,
    text=rate_df["Placement rate"].round(1),
    category_orders={"Score band": band_labels},
    hover_data=["Students"],
    title="Placement rate rises as scores improve"
)

fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(showlegend=False, height=750)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

This chart helps you tell a real story:

1. SSC %, HSC %, and degree % show a clear upward pattern.
2. Students with stronger academic history are much more likely to be placed.
3. MBA % looks weaker as a placement factor.
4. Employability test % matters, but not as clearly as earlier academic performance.

So one likely conclusion is:

Placement seems to depend more on long-term academic consistency than on MBA percentage alone.

In [42]:
long_scores = df.melt(
    id_vars="status",
    value_vars=score_cols,
    var_name="Feature",
    value_name="Score"
)

long_scores["Feature"] = long_scores["Feature"].map(labels)

fig = px.box(
    long_scores,
    x="Feature",
    y="Score",
    color="status",
    points="all",
    title="Placed students tend to have stronger academic profiles"
)

fig.show()

The score distributions for placed students are generally shifted upward, especially for school and degree percentages. This supports the idea that academic performance is associated with placement.

In [43]:
cat_features = ["workex", "specialisation", "degree_t", "hsc_s"]

cat_frames = []

for col in cat_features:
    temp = (
        df.groupby(col)["placed"]
        .mean()
        .mul(100)
        .reset_index(name="Placement rate")
    )
    temp["Feature"] = labels[col]
    temp = temp.rename(columns={col: "Category"})
    cat_frames.append(temp)

cat_df = pd.concat(cat_frames, ignore_index=True)

fig = px.bar(
    cat_df,
    x="Category",
    y="Placement rate",
    facet_col="Feature",
    facet_col_wrap=2,
    text=cat_df["Placement rate"].round(1),
    title="Placement differences across categorical groups"
)

fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(showlegend=False, height=700)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

This chart is very useful because it may show:

1. Work experience gives a strong advantage.
2. Mkt&Fin may have better placement than Mkt&HR.
3. Some degree types may lead to stronger placement outcomes.
4. School board may not matter much.

That means your story becomes more interesting:

Placement is not only about marks. Work experience and specialisation also create visible differences.

In [44]:
heat = (
    df.groupby(["workex", "specialisation"])["placed"]
    .mean()
    .mul(100)
    .reset_index(name="Placement rate")
)

heat_pivot = heat.pivot(
    index="workex",
    columns="specialisation",
    values="Placement rate"
).reindex(index=["No", "Yes"])

fig = go.Figure(data=go.Heatmap(
    z=heat_pivot.values,
    x=heat_pivot.columns,
    y=heat_pivot.index,
    text=np.round(heat_pivot.values, 1),
    texttemplate="%{text}%",
    colorscale="Blues",
    hovertemplate="Work experience: %{y}<br>Specialisation: %{x}<br>Placement rate: %{z:.1f}%<extra></extra>"
))

fig.update_layout(
    title="Work experience and specialisation together matter",
    xaxis_title="Specialisation",
    yaxis_title="Work experience"
)

fig.show()

Students with work experience and a finance specialisation appear to have the highest placement rates, while students without work experience and with HR specialisation appear to face the most difficulty.

In [45]:
fig = px.histogram(
    placed_df,
    x="salary",
    nbins=25,
    title="Salary distribution among placed students"
)

fig.show()

Salary is right-skewed, meaning most students receive moderate salary offers, while a smaller number receive much higher offers.

In [46]:
num_cols = ["ssc_p", "hsc_p", "degree_p", "etest_p", "mba_p", "salary"]
corr_df = placed_df[num_cols].corr().round(2)

fig = px.imshow(
    corr_df,
    text_auto=True,
    aspect="auto",
    title="Correlation between numeric features (placed students only)"
)
fig.show()

Salary has only weak-to-moderate correlation with academic percentages, suggesting that salary is influenced by more than academic marks alone.

In [47]:
fig = px.box(
    placed_df,
    x="specialisation",
    y="salary",
    color="workex",
    points="all",
    title="Among placed students, salary is higher for students with work experience"
)

fig.show()

Work experience appears to be related not only to placement itself, but also to higher salary offers. In addition, finance specialisation may be linked to better salary outcomes than HR

In [48]:
fig = px.box(
    placed_df,
    x="degree_t",
    y="salary",
    color="specialisation",
    points="all",
    title="Salary differences by degree type and specialisation"
)

fig.show()

Degree type may also play a role in salary. Some fields seem to receive higher offers, especially when combined with certain specialisations.

In [49]:
fig = px.scatter(
    placed_df,
    x="etest_p",
    y="salary",
    color="specialisation",
    symbol="workex",
    size="mba_p",
    hover_data=["degree_t", "ssc_p", "hsc_p", "degree_p"],
    title="Employability test and salary"
)

fig.show()

The employability test may have some positive relationship with salary, but the pattern is not extremely strong. This suggests that salary is influenced by multiple factors, not just one test score.

The main story of the dataset is that placement is associated most strongly with academic consistency, work experience, and specialisation. Earlier academic percentages such as SSC, HSC, and degree performance appear to separate placed from not placed students more clearly than MBA percentage does. Work experience gives an additional advantage, and the combination of work experience with finance specialisation seems especially favorable. Among students who are placed, salary differences still exist, and these seem to be related more to work experience, specialisation, and degree type than to one single academic score.

## Key Findings

1. Placement is positively associated with stronger SSC, HSC, and degree percentages.
2. Work experience appears to improve both placement chances and salary outcomes.
3. Students in Mkt&Fin seem to perform better in placement and salary than students in Mkt&HR.
4. MBA percentage appears less influential for placement than earlier academic performance.